# Invoice OCR and tariff classification

This notebook calls the shared `AI` package. It has no separate classifier definitions or Colab-only imports. See `AI/README.md` for installation, the processing flow, configuration, outputs, and limitations.


## Locate the repository

Open this notebook from the checkout. In Colab, clone or mount the checkout first and set `REPO_ROOT` if necessary.


In [ ]:
from pathlib import Path
import os
import sys

# Run from the repository root or AI/. Set REPO_ROOT if the notebook is elsewhere.
configured_root = os.environ.get("REPO_ROOT")
candidates = [Path(configured_root)] if configured_root else [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((root for root in candidates if (root / "AI" / "main.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Set REPO_ROOT to the checkout containing AI/main.py")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)


## Install dependencies

Use a CUDA-enabled Python environment. Install a matching PyTorch and torchvision CUDA build for your host, then run the optional cell below. Restart the kernel after changing dependencies.


In [ ]:
# Optional installation in this notebook kernel:
# import subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "AI" / "requirements.txt")])


## Configure the document and retrieval backend

Set `PDF_PATH` to your PDF. Catalog mode uses a local CSV or downloads the public catalog at runtime. National codes require a matching overlay. Weaviate mode searches the configured country-specific database instead. Models load only when the pipeline is called.


In [ ]:
from uuid import uuid4
from AI.main import doc_pipeline

PDF_PATH = Path(os.environ.get("PDF_PATH", "invoice.pdf")).expanduser().resolve()
OUTPUT_DIR = REPO_ROOT / "output" / f"{PDF_PATH.stem}_{uuid4().hex}"
BACKEND = "catalog"  # or "weaviate" with WEAVIATE_* environment variables configured
CATALOG_PATH = os.environ.get("HS_CATALOG_PATH")  # local HS CSV, or download on run
NATIONAL_PATH = os.environ.get("NATIONAL_INDEX_PATH")  # optional national overlay JSON


## Run the pipeline

PDF pages → raw OCR → cleaned text → validated invoice JSON → retrieved tariff candidates → constrained classification. Every invoice and line item is processed. Each run uses its own directory.


In [ ]:
result = doc_pipeline(
    PDF_PATH, OUTPUT_DIR, backend=BACKEND,
    catalog_path=CATALOG_PATH, national_path=NATIONAL_PATH,
    progress=print,
)
print("Results:", OUTPUT_DIR)


## Inspect every invoice

`hs6_code` is the inferred international code. National fields stay null without matching national candidates. Codes explicitly stated on the invoice are retained as evidence and flagged for review. Read `validation_note` and `needs_review` before using a result.


In [ ]:
from IPython.display import JSON, display

for invoice_index, invoice in enumerate(result["invoices"], 1):
    print("Invoice", invoice_index, invoice["order_level"]["invoice_number"])
    for item in invoice["item_level"]:
        print(item.get("sku_code"), item.get("hs6_code"),
              item.get("destination_country_hts_code"),
              "review:", item.get("needs_review"))
display(JSON(result))
